# Orchestrating the Inference Engine Core

In our journey to understand vLLM, we've met the individual players: the `Scheduler` who plans the work and the `Executor` who carries it out. Now, we need the conductor who brings them together to make music. This notebook introduces the `EngineCore`, the central orchestrator that coordinates all the moving parts of the inference system.

By the end of this notebook, you will understand how the `EngineCore` acts as the central hub for processing requests. You will see how it directs the flow of data from the `InputProcessor`, through the `Scheduler` and `Executor`, and finally to the `OutputProcessor`. You will build a simplified version of this core orchestrator yourself, solidifying your mental model of how a high-throughput LLM serving system operates.

In [ ]:
# Full Setup Cell

import collections
from typing import List, Dict, Any, NamedTuple

# --- Data Structures ---
# Represents a user's request
class Request:
    def __init__(self, request_id: str, prompt: str, max_tokens: int):
        self.request_id = request_id
        self.prompt = prompt
        self.output_tokens = []
        self.max_tokens = max_tokens
        self.finished = False

    def __repr__(self):
        return f"Request({self.request_id}, tokens={len(self.output_tokens)}/{self.max_tokens}, finished={self.finished})"

# Represents the plan for what to run in one model iteration
class SchedulerOutput(NamedTuple):
    requests_to_run: List[Request]
    requests_to_finish: List[Request]

# Represents the raw output from the model execution
class ModelOutput(NamedTuple):
    next_tokens: Dict[str, str]  # {request_id: next_token}

# Represents the final, formatted output for finished requests
class EngineOutput(NamedTuple):
    request_id: str
    generated_text: str

# --- Mock Components ---

class MockInputProcessor:
    """Prepares raw requests for the engine."""
    def process(self, request_id: str, prompt: str, max_tokens: int) -> Request:
        print(f"[InputProcessor] Processing request {request_id}")
        return Request(request_id, prompt, max_tokens)

class MockScheduler:
    """Decides which requests to run and manages their state."""
    def __init__(self):
        self.request_queue = collections.deque()

    def add_request(self, req: Request):
        print(f"[Scheduler]      Adding {req.request_id} to the queue.")
        self.request_queue.append(req)

    def schedule(self) -> SchedulerOutput:
        print("[Scheduler]      Scheduling next batch...")
        if not self.request_queue:
            return SchedulerOutput([], [])

        # Simple logic: run all requests in the queue.
        # In vLLM, this is where continuous batching happens.
        batch = list(self.request_queue)
        print(f"[Scheduler]      Batch created with {len(batch)} requests.")
        return SchedulerOutput(requests_to_run=batch, requests_to_finish=[])

    def update_from_output(self, model_output: ModelOutput) -> List[Request]:
        """Updates request state based on model output."""
        finished_requests = []
        # We need to iterate over a copy as we might modify the original queue
        for req in list(self.request_queue):
            if req.request_id in model_output.next_tokens:
                new_token = model_output.next_tokens[req.request_id]
                req.output_tokens.append(new_token)
                print(f"[Scheduler]      Updating {req.request_id} with token '{new_token}'.")

                # Check if the request is finished
                if len(req.output_tokens) >= req.max_tokens:
                    req.finished = True
                    finished_requests.append(req)
                    self.request_queue.remove(req)
                    print(f"[Scheduler]      Request {req.request_id} is finished.")
        return finished_requests

class MockExecutor:
    """Runs the model on the hardware."""
    def execute_model(self, scheduler_output: SchedulerOutput) -> ModelOutput:
        print(f"[Executor]       Executing model for {len(scheduler_output.requests_to_run)} requests...")
        # Simulate generating a token for each request in the batch
        next_tokens = {}
        for req in scheduler_output.requests_to_run:
            # Fake token generation
            next_token = f"t{len(req.output_tokens) + 1}"
            next_tokens[req.request_id] = next_token
        print(f"[Executor]       Generated tokens: {next_tokens}")
        return ModelOutput(next_tokens=next_tokens)

class MockOutputProcessor:
    """Formats the final output for the user."""
    def process(self, finished_requests: List[Request]) -> List[EngineOutput]:
        outputs = []
        for req in finished_requests:
            generated_text = "".join(req.output_tokens)
            print(f"[OutputProcessor] Formatting output for {req.request_id}: '{generated_text}'")
            outputs.append(EngineOutput(req.request_id, generated_text))
        return outputs

### The Core Idea: A Restaurant Kitchen's Head Chef

Imagine a busy, high-end restaurant kitchen. It's a complex system with many moving parts. How does it all work without descending into chaos? Through careful coordination, led by the Head Chef.

*   **Maître d' (`InputProcessor`):** Greets customers and takes their orders. They translate "I'd like the steak, medium-rare" into a standardized ticket for the kitchen (`Request` object).
*   **Sous-Chef (`Scheduler`):** Manages the cooking line. They look at all the incoming tickets, group them intelligently (e.g., "these three tables all ordered fries, let's do one big batch"), and decide what the line cooks should work on *right now*. This is scheduling and batching.
*   **Line Cooks (`Executor`):** These are the experts at the grill, fryer, and sauté stations. They take a batch of tasks from the Sous-Chef and execute them perfectly, cooking the food. This is the model forward pass on the GPU.
*   **Food Expediter (`OutputProcessor`):** Takes the cooked components from the line cooks, assembles the final plate, adds garnish, and makes sure it looks perfect before it goes to the customer. This is formatting the final output.

And the **Head Chef (`EngineCore`)**? The Head Chef doesn't cook. Instead, they stand at the center of the kitchen, orchestrating everything. They create the main "loop" of work:

1.  They watch the Sous-Chef (`Scheduler`) create a plan for the next batch.
2.  They hand that plan to the Line Cooks (`Executor`) to execute.
3.  They take the cooked food from the Line Cooks and give it back to the Sous-Chef to update their status board (e.g., "fries for table 5 are done").
4.  If a dish is complete, they pass it to the Food Expediter (`OutputProcessor`) for final presentation.

The `EngineCore` is this Head Chef. It runs the main processing loop, ensuring each component does its job in the right order, turning a queue of raw requests into a stream of finished outputs.

In [ ]:
class MiniEngineCore:
    """The central orchestrator for the inference engine."""

    def __init__(
        self,
        scheduler: MockScheduler,
        executor: MockExecutor,
    ):
        self.scheduler = scheduler
        self.executor = executor
        print("[EngineCore]     Initialized. Ready to orchestrate.")

    def add_request(self, req: Request):
        """Adds a new request to the scheduler's queue."""
        self.scheduler.add_request(req)

    def step(self) -> List[Request]:
        """
        Performs one iteration of the engine loop.
        This is the heartbeat of the system.
        Returns a list of requests that finished in this step.
        """
        print("\n--- EngineCore: Starting Step ---")

        # 1. Ask the Scheduler for a plan.
        scheduler_output = self.scheduler.schedule()

        if not scheduler_output.requests_to_run:
            print("[EngineCore]     No requests to run in this step.")
            return []

        # 2. Give the plan to the Executor to run the model.
        model_output = self.executor.execute_model(scheduler_output)

        # 3. Tell the Scheduler what happened so it can update state.
        finished_requests = self.scheduler.update_from_output(model_output)

        print(f"[EngineCore]     Step finished. {len(finished_requests)} requests completed.")
        return finished_requests

Let's walk through our `MiniEngineCore` implementation line by line.

*   **`__init__(self, scheduler, executor)`**: The constructor is simple. The `EngineCore`'s primary role is coordination, so it just needs references to the components it needs to coordinate. It holds onto the `scheduler` and `executor` so it can call their methods.

*   **`add_request(self, req)`**: This is a convenience method. Instead of users talking directly to the `Scheduler`, they can go through the `EngineCore`, which simply passes the request along. This maintains the `EngineCore` as the central point of interaction.

*   **`step()`**: This is the most important method. It represents a single "heartbeat" or "tick" of the entire inference engine. In a real system, this method is called in a tight loop.
    1.  `scheduler_output = self.scheduler.schedule()`: The first thing the `EngineCore` does is ask the `Scheduler`, "What work should we do *right now*?". The `Scheduler` looks at all waiting and running requests and produces a `SchedulerOutput`, which is a concrete plan for the next model forward pass.
    2.  `model_output = self.executor.execute_model(scheduler_output)`: The `EngineCore` takes this plan and hands it to the `Executor`, saying, "Here, run the model on this batch." The `Executor` does the heavy lifting on the GPU and returns the results—in our case, the next token for each request in the batch.
    3.  `finished_requests = self.scheduler.update_from_output(model_output)`: Finally, the `EngineCore` takes the results from the `Executor` and reports back to the `Scheduler`. It says, "The model produced these tokens. Update your records." The `Scheduler` updates the state of each request, appends the new tokens, and determines if any requests have now met their `max_tokens` limit and can be considered finished.

The `step` function returns the list of requests that were completed during this iteration, which can then be passed to an `OutputProcessor`. This clean, three-step loop is the fundamental operating principle of the entire engine.

In [ ]:
# --- Demonstration ---

# 1. Initialize all our components
input_processor = MockInputProcessor()
scheduler = MockScheduler()
executor = MockExecutor()
output_processor = MockOutputProcessor()

# 2. Initialize the EngineCore, our orchestrator
engine_core = MiniEngineCore(scheduler, executor)

# 3. Create and add some requests
req1_raw = ("req-1", "Hello", 3)
req2_raw = ("req-2", "My name is", 4)

req1 = input_processor.process(*req1_raw)
req2 = input_processor.process(*req2_raw)

engine_core.add_request(req1)
engine_core.add_request(req2)

# 4. Run the engine loop until all requests are finished
all_finished_outputs = []
max_steps = 5
step_num = 0

# The scheduler has requests as long as its queue is not empty
while scheduler.request_queue and step_num < max_steps:
    step_num += 1
    # Run one step of the engine
    finished_requests_this_step = engine_core.step()

    # Process the outputs of any requests that finished
    if finished_requests_this_step:
        final_outputs = output_processor.process(finished_requests_this_step)
        all_finished_outputs.extend(final_outputs)

print("\n--- All requests processed! ---")
print("Final outputs:", all_finished_outputs)

### Going Deeper: The Power of the `step()` Abstraction

A key design decision in vLLM's `EngineCore` is its "heartbeat" loop, embodied by the `step()` function. You might wonder, why not have a simpler function like `process_request(request)` that takes a single request and runs it from start to finish?

The answer is **throughput**.

A `process_request` function would be inefficient. It would take one request, run the model to generate a token, then loop to generate the next token, and so on, until that single request is done. The entire GPU would be dedicated to one request at a time. This is called **iterative decoding**.

The `step()` function enables a much more powerful paradigm: **continuous batching**.

At every single `step`, the `Scheduler` can look at *all* requests currently in the system—some that are brand new (prefill stage) and some that are part-way through generation (decode stage). It can bundle them all into a single, large batch to be processed by the `Executor` in one parallel forward pass.

*   **Step 1:** Request A and B arrive. The scheduler batches them for their first token.
*   **Step 2:** Request C arrives. The scheduler batches A, B, and C together. A and B need their second token, C needs its first.
*   **Step 3:** Request A finishes. The scheduler batches B and C for their next tokens.

This approach keeps the GPU constantly busy with large batches, dramatically increasing the number of requests that can be handled per second (throughput). The `EngineCore`'s `step()` loop is the mechanism that drives this continuous, dynamic batching process. It decouples the engine's operation from the lifecycle of any single request, allowing it to think in terms of system-wide optimization at every single tick.

In [ ]:
# --- Visualization of the EngineCore Loop ---

def visualize_engine_flow():
    """Prints an ASCII art diagram of the EngineCore's control flow."""
    art = r"""
        [New Requests]
              |
              v
    [InputProcessor] --> [Request Objects]
                                |
                                v
    +-------------------- [Scheduler Queue] ----------------------+
    |                                                             |
    |   +-----------------------------------------------------+   |
    |   |                   ENGINE CORE                       |   |
    |   |                                                     |   |
    |   |   1. schedule()  +------------+   2. execute_model()  |   |
    |   |  --------------> | Scheduler  | --------------------+   |
    |   |                  +------------+                     |   |
    |   |                        ^                            v   |
    |   |                        |                      +------------+
    |   |   3. update_from_output()                     |  Executor  |
    |   |                        +----------------------+------------+
    |   |                                                     |
    |   +-----------------------------------------------------+   |
    |                                                             |
    +--> [Finished Requests] --> [OutputProcessor] --> [Final Output]

    The EngineCore continuously runs the loop (1 -> 2 -> 3),
    orchestrating the flow of requests between the Scheduler and Executor.
    """
    print(art)

visualize_engine_flow()

### Summary

In this notebook, we built `MiniEngineCore`, the central nervous system of our inference engine. We saw how it doesn't perform the low-level tasks itself but instead orchestrates the components that do, creating a seamless flow from request intake to final output.

**What we built:**
*   A simplified `EngineCore` class that manages the main inference loop.
*   A `step()` method that embodies one "heartbeat" of the engine, coordinating the `Scheduler` and `Executor`.
*   A demonstration showing how multiple requests are processed step-by-step through this orchestrated loop.

**Key Takeaways:**
*   **Orchestration over Implementation:** The `EngineCore`'s value is in coordination. It connects the planning (`Scheduler`) and action (`Executor`) components.
*   **The "Heartbeat" Loop:** The `step()` function is the core of the engine's operation, enabling powerful strategies like continuous batching by processing work in discrete, batch-oriented iterations.
*   **Centralized Control:** By routing interactions through the `EngineCore`, the system becomes more modular and easier to manage.

**What's Next:**
The `EngineCore` is the powerful, internal engine. But how do users interact with it? In the next notebook, `The User-Facing LLM Engine API`, we will build the `LLMEngine`, a higher-level API that wraps the `EngineCore` to provide a clean, easy-to-use interface for submitting prompts and receiving generated text, hiding the complexity of the internal `step()` loop from the end-user.